In [4]:
import polars as pl
import numpy as np

import nltk
import textwrap
from nltk import sent_tokenize
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

nltk.download("punkt", "../datasets/nltk/")
nltk.download("stopwords", "../datasets/nltk/")

nltk.data.path.append("../datasets/nltk/")

[nltk_data] Downloading package punkt to ../datasets/nltk/...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to ../datasets/nltk/...
[nltk_data]   Package stopwords is already up-to-date!


In [49]:
class TextRank:
    def __init__(self, csv_path: str, filter: str = "business"):
        self.documents = pl.read_csv(csv_path)
        self.documents = self._filter_by_label(filter)

        self.featurizer = TfidfVectorizer(stop_words = stopwords.words("english"),
                                          norm = "l1")

    def _filter_by_label(self, label: str | None):
        if not label:
            return self.documents
        return self.documents.filter( pl.col("labels") == label )

    def __getitem__(self, idx: int):
        #print("#"*200)
        #print(self.wrap(self.documents[idx]))
        #print("#"*200)
        self.summerize(self.documents[idx])

    def wrap(self, x):
        return textwrap.fill(x["text"][0], replace_whitespace = False, fix_sentence_endings = True)

    def summerize(self, doc: str):
        _, text = self._split_title_text(doc)

        sents = sent_tokenize(text)
        x = self.featurizer.fit_transform(sents)

        scores = self.get_scores(x)
        sort_idx = np.argsort(-scores)

        self.generate_summary(sents, sort_idx)

    def _split_title_text(self, document: pl.Series) -> tuple[str, str]:
        return document["text"][0].split("\n", 1)

    def get_scores(self, x ):
        G = cosine_similarity(x)
        G = self.normalize(G)

        A = self.pagerank_matrix(G)
        return self.limiting_stationary_distribution(A)

    def normalize(self, G):
        return np.divide(G, G.sum(axis = 1, keepdims = True))

    def pagerank_matrix(self, G, factor = 0.15):
        U = np.ones_like(G) / len(G)
        return (1 - factor) * G + factor * U

    def limiting_stationary_distribution(self, A):
        eigenvals, eigenvecs = np.linalg.eig(A.T)

        limiting_dist = np.ones(len(A)) / len(A)
        threshold = 1e-8
        delta = float('inf')
        iters = 0

        while delta > threshold:
            iters += 1

            p = limiting_dist.dot(A)

            delta = np.abs(p - limiting_dist).sum()

            limiting_dist = p
        
        return limiting_dist


    def generate_summary(self, sents, indexes):
        for i in indexes[:5]:
            print(sents[int(i)])


In [51]:
TextRank("../datasets/bbc_text_cls.csv")[100]

The Bank said it had acted to curb inflation but the move was criticised by some analysts.
However, opposition parties and some analysts said the move was ill-timed given data showing the Australian economy grew just 0.1% between October and December and 1.5% on an annual basis.
"That 1.5% annual growth rate is the lowest we have seen since the post-election slump we saw back in 2000-1," said Michael Blythe, chief economist at the Commonwealth Bank of Australia.
The Reserve Bank of Australia lifted interest rates 0.25% to 5.5%, their first upwards move in more than a year.
The Australian government said the economy remained strong with unemployment at a near 30 year low.


In [2]:
from sumy.summarizers.text_rank import TextRankSummarizer
from sumy.summarizers.lsa import LsaSummarizer
from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer

In [18]:
class SumyTextRank:
    def __init__(self, csv_path: str, filter: str = "business"):
        self.documents = pl.read_csv(csv_path)
        self.documents = self._filter_by_label(filter)

        self.summerizer = TextRankSummarizer()

    def _filter_by_label(self, label: str | None):
        if not label:
            return self.documents
        return self.documents.filter( pl.col("labels") == label )

    def __getitem__(self, idx: int):
        #print("#"*200)
        #print(self.wrap(self.documents[idx]))
        #print("#"*200)
        self.summerize(self.documents[idx])

    def wrap(self, x):
        return textwrap.fill(x, replace_whitespace = False, fix_sentence_endings = True)

    def summerize(self, doc: str):
        _, text = self._split_title_text(doc)

        parser = PlaintextParser.from_string(
            text, Tokenizer("english")
        )

        self.generate_summary(parser, 5)

    def _split_title_text(self, text: str) -> tuple[str, str]:
        return text["text"][0].split("\n", 1)

    def get_scores(self, x ):
        G = cosine_similarity(x)
        G = self.normalize(G)

        A = self.pagerank_matrix(G)
        return self.limiting_stationary_distribution(A)

    def normalize(self, G):
        return np.divide(G, G.sum(axis = 1, keepdims = True))

    def pagerank_matrix(self, G, factor = 0.15):
        U = np.ones_like(G) / len(G)
        return (1 - factor) * G + factor * U

    def limiting_stationary_distribution(self, A):
        eigenvals, eigenvecs = np.linalg.eig(A.T)

        limiting_dist = np.ones(len(A)) / len(A)
        threshold = 1e-8
        delta = float('inf')
        iters = 0

        while delta > threshold:
            iters += 1

            p = limiting_dist.dot(A)

            delta = np.abs(p - limiting_dist).sum()

            limiting_dist = p
        
        return limiting_dist


    def generate_summary(self, parser, sentences_count):
        
        for s in self.summerizer(parser.document, sentences_count):
            print(self.wrap(str(s)))

In [19]:
SumyTextRank("../datasets/bbc_text_cls.csv")[100]

However, shortly after the Bank made its decision, new figures showed
a fall in economic growth in the last quarter.
"Over recent months it has become increasingly clear that remaining
spare capacity in the labour and goods markets is becoming rather
limited," said Ian Macfarlane, Governor of the Reserve Bank.
However, exports declined in the second half of 2004, fuelling a rise
in the country's current account deficit - the difference in the value
of imports compared to exports - to a record Australian dollar 29.4bn.
Stock markets had factored in the likelihood of a rate rise but
analysts still expressed concern about the strength of the economy.
"That 1.5% annual growth rate is the lowest we have seen since the
post-election slump we saw back in 2000-1," said Michael Blythe, chief
economist at the Commonwealth Bank of Australia.
